In [ ]:
# Mount Drive & Imports
from google.colab import drive
drive.mount('/content/drive')

import os, cv2, numpy as np, shutil
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import VGG16
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

# Paths
PROJECT_DIR = '/content/drive/MyDrive/CIS515FinalProject'
BOX_DIR = os.path.join(PROJECT_DIR, 'boxes')
DATA_DIR    = os.path.join(PROJECT_DIR, 'data')  # will get open/ & closed/

# Make output dirs
for cls in ("open","closed"):
    os.makedirs(os.path.join(DATA_DIR, cls), exist_ok=True)

# Auto‐label & copy each box image
file_paths, labels = [], []
for fname in sorted(os.listdir(BOX_DIR)):
    if not fname.lower().endswith('.png'):
        continue
    src = os.path.join(BOX_DIR, fname)
    img = cv2.imread(src)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    # define HSV masks as uint8
    r1_lo, r1_hi = np.array([  0, 70, 50],np.uint8), np.array([ 10,255,255],np.uint8)
    r2_lo, r2_hi = np.array([170, 70, 50],np.uint8), np.array([180,255,255],np.uint8)
    b_lo,  b_hi  = np.array([100,150,  0],np.uint8), np.array([140,255,255],np.uint8)

    mask_r = cv2.inRange(hsv, r1_lo, r1_hi) + cv2.inRange(hsv, r2_lo, r2_hi)
    mask_b = cv2.inRange(hsv, b_lo,  b_hi)

    label_str = 'closed' if cv2.countNonZero(mask_r) > cv2.countNonZero(mask_b) else 'open'
    dst       = os.path.join(DATA_DIR, label_str, fname)
    shutil.copy(src, dst)

    file_paths.append(dst)
    labels.append(1 if label_str=='closed' else 0)

# Dataset summary
total  = len(labels)
opens  = labels.count(0)
closes = labels.count(1)
print(f"Raw images: {total}  → OPEN={opens}, CLOSED={closes}")

# Stratified split
train_p, val_p, train_l, val_l = train_test_split(
    file_paths, labels, test_size=0.2, stratify=labels, random_state=42
)
print(f"Train: {len(train_l)}  (OPEN={train_l.count(0)}, CLOSED={train_l.count(1)})")
print(f" Val : {len(val_l)}  (OPEN={val_l.count(0)}, CLOSED={val_l.count(1)})\n")

# tf.data pipeline
IMG_SIZE=(224,224); BATCH=8

def preprocess(path, label):
    img = tf.io.read_file(path)
    img = tf.io.decode_png(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE) / 255.0
    label = tf.cast(label, tf.float32)
    return img, label

def make_ds(paths, labs, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((paths,labs))
    if shuffle:
        ds = ds.shuffle(len(paths), seed=42)
    return ds.map(preprocess, tf.data.AUTOTUNE) \
             .batch(BATCH) \
             .prefetch(tf.data.AUTOTUNE)

train_ds = make_ds(train_p, train_l)
val_ds   = make_ds(val_p,   val_l, shuffle=False)

# Model definition
base = VGG16(weights='imagenet', include_top=False, input_shape=IMG_SIZE+(3,))
base.trainable = False

model = models.Sequential([
    base,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid'),
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

# Train & history
EPOCHS = 5
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS)

print("\nPer-epoch Training Stats:")
for e in range(EPOCHS):
    print(f" Epoch {e+1}: "
          f"loss={history.history['loss'][e]:.3f}, "
          f"acc={history.history['accuracy'][e]:.3f}, "
          f"val_loss={history.history['val_loss'][e]:.3f}, "
          f"val_acc={history.history['val_accuracy'][e]:.3f}"
    )

# Final Evaluation
loss, acc = model.evaluate(val_ds, verbose=0)
print(f"\nFinal Val loss: {loss:.4f}")
print(f"Final Val accuracy: {acc:.4f}\n")

# detailed report
y_true  = np.array(val_l)
y_predp = model.predict(val_ds).flatten()
y_pred  = (y_predp > 0.5).astype(int)

print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=['OPEN','CLOSED']))
print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Raw images: 30  → OPEN=11, CLOSED=19
Train: 24  (OPEN=9, CLOSED=15)
 Val : 6  (OPEN=2, CLOSED=4)



Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vgg16 (Functional)              │ (None, 7, 7, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,846,273 (56.63 MB)

 Trainable params: 131,585 (514.00 KB)

 Non-trainable params: 14,714,688 (56.13 MB)

Epoch 1/5
3/3 ━━━━━━━━━━━━━━━━━━━━ 37s 10s/step - accuracy: 0.3229 - loss: 0.8847 - val_accuracy: 0.5000 - val_loss: 0.6832
Epoch 2/5
3/3 ━━━━━━━━━━━━━━━━━━━━ 27s 8s/step - accuracy: 0.6562 - loss: 0.6764 - val_accuracy: 0.6667 - val_loss: 0.6901
Epoch 3/5
3/3 ━━━━━━━━━━━━━━━━━━━━ 26s 9s/step - accuracy: 0.6719 - loss: 0.5761 - val_accuracy: 0.6667 - val_loss: 0.7165
Epoch 4/5
3/3 ━━━━━━━━━━━━━━━━━━━━ 36s 7s/step - accuracy: 0.7760 - loss: 0.5185 - val_accuracy: 0.5000 - val_loss: 0.7477
Epoch 5/5
3/3 ━━━━━━━━━━━━━━━━━━━━ 19s 7s/step - accuracy: 0.7292 - loss: 0.4985 - val_accuracy: 0.5000 - val_loss: 0.7739

Per-epoch Training Stats:
 Epoch 1: loss=0.857, acc=0.333, val_loss=0.683, val_acc=0.500
 Epoch 2: loss=0.670, acc=0.625, val_loss=0.690, val_acc=0.667
 Epoch 3: loss=0.551, acc=0.750, val_loss=0.717, val_acc=0.667
 Epoch 4: loss=0.562, acc=0.708, val_loss=0.748, val_acc=0.500
 Epoch 5: loss=0.531, acc=0.708, val_loss=0.774, val_acc=0.500

Final Val loss: 0.7739
Final Val accuracy